# ✅ GIADA Task 3c v2 — Conferma degli esatti checkpoint Task 3b
Nessun riaddestramento: verifichiamo e riusiamo i tre checkpoint simmetrici già congelati prima del fresh.

In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_3c_v2'); GIADA_REPO=WORK/'giada'; TEACHER_REPO=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip(); print({'revision':REVISION})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]: del sys.modules[name]
import torch
assert torch.cuda.is_available(),'La conferma Task 3c richiede una GPU CUDA Kaggle.'
from src.giada_teacher import ExtractedGateFormula,JointGateSymmetricConfirmationConfig,prepare_joint_gate_symmetric_confirmation,freeze_task3c_from_task3b,evaluate_frozen_symmetric_joint_gate
from src.giada_teacher.joint_gate_symmetric_confirmation import EXPECTED_TASK3B_ARCHIVE_SHA256,EXPECTED_TASK3B_REPORT_SHA256
prereg=json.loads((GIADA_REPO/'experiments/teacher_joint_gate_symmetric_confirmation_preregistration_v2.json').read_text())
display({'gpu':torch.cuda.get_device_name(0),'preregistration':prereg})


In [ ]:
def file_sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''): digest.update(chunk)
    return digest.hexdigest()
INPUT_ROOT=Path('/kaggle/input'); override=os.environ.get('GIADA_TASK3B_ARTIFACT')
candidates=[Path(override).expanduser()] if override else []
if INPUT_ROOT.is_dir():
    candidates += list(INPUT_ROOT.rglob('giada_joint_m_h_optimization_diagnosis.zip'))
    candidates += list(INPUT_ROOT.rglob('archive.zip'))
    candidates += [p.parent for p in INPUT_ROOT.rglob('final_report.json') if (p.parent/'diagnostic_checkpoints.pt').is_file()]
def exact_source(path):
    try:
        if path.is_file(): return file_sha(path)==EXPECTED_TASK3B_ARCHIVE_SHA256
        return file_sha(path/'final_report.json')==EXPECTED_TASK3B_REPORT_SHA256 and (path/'diagnostic_checkpoints.pt').is_file()
    except Exception: return False
TASK3B_SOURCE=next((p.resolve() for p in candidates if p.exists() and exact_source(p)),None)
assert TASK3B_SOURCE is not None,'Artefatto esatto giada_joint_m_h_optimization_diagnosis non trovato negli Input Kaggle.'
print({'task3b_source':str(TASK3B_SOURCE)})


In [ ]:
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_joint_m_h_symmetric_confirmation_v2')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
bundle=prepare_joint_gate_symmetric_confirmation(formula); config=JointGateSymmetricConfirmationConfig()
display({'contract':bundle['confirmation_contract'],'seeds':config.seeds,'fixed_checkpoint':config.checkpoints[-1]})


## 🔒 Verifica e freeze senza training
La cella verifica SHA-256, riproduce gli score development e congela gli esatti pesi Task 3b.

In [ ]:
freeze_report=freeze_task3c_from_task3b(bundle,OUTPUT_DIR,TASK3B_SOURCE,config,code_revision=REVISION)
display({'valid':freeze_report['valid'],'source_verified':freeze_report['source_verified'],'retraining_performed':freeze_report['retraining_performed'],'development':freeze_report['development'],'fresh_accessed':freeze_report['fresh_accessed']})
assert freeze_report['valid'] and freeze_report['source_verified'] and not freeze_report['retraining_performed'] and not freeze_report['fresh_accessed']


## 🧪 Apertura fresh una sola volta
Eseguire soltanto dopo il freeze valido della cella precedente.

In [ ]:
final=evaluate_frozen_symmetric_joint_gate(bundle,OUTPUT_DIR,config)
display({'valid':final['valid'],'decision':final['decision'],'selection_used_fresh':final['selection_used_fresh']})
assert final['valid'] and not final['selection_used_fresh']


## 📦 Download
Download Blob/base64 compatibile con Kaggle.

In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_joint_m_h_symmetric_confirmation_v2','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
